# Olist Brazilian E-Commerce — End-to-End Analysis

**Dataset:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)  
**Period:** 2016–2018 | ~100k orders across 8 relational tables

## Executive Summary
> *(Fill in after analysis)*

## Table of Contents
1. [Setup & Data Loading](#1-setup--data-loading)
2. [Schema Exploration](#2-schema-exploration)
3. [Data Cleaning](#3-data-cleaning)
4. [Exploratory Data Analysis](#4-exploratory-data-analysis)
5. [Angle 1 — Delivery Performance](#5-angle-1--delivery-performance)
6. [Angle 2 — Customer Segmentation (RFM)](#6-angle-2--customer-segmentation-rfm)
7. [Angle 3 — Review Score Prediction](#7-angle-3--review-score-prediction)
8. [Findings & Recommendations](#8-findings--recommendations)

## 1. Setup & Data Loading

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

from src.load import load_raw, build_master
from src.clean import clean, filter_delivered

sns.set_theme(style='whitegrid', palette='muted')
OUTPUTS = '../outputs/'

dfs = load_raw()

## 2. Schema Exploration

In [ ]:
# Quick audit: shape, dtypes, null counts for each table
for name, df in dfs.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print(f'\n=== {name} === {df.shape}')
    print(df.dtypes.to_string())
    if len(nulls):
        print('Nulls:', nulls.to_dict())

In [ ]:
# Referential integrity check: orphan order_ids
order_ids = set(dfs['olist_orders_dataset']['order_id'])
for name in ['olist_order_items_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset']:
    orphans = set(dfs[name]['order_id']) - order_ids
    print(f'{name}: {len(orphans)} orphan order_ids')

## 3. Data Cleaning

In [ ]:
master = build_master(dfs)
master = clean(master)
print(f'\nMaster shape: {master.shape}')
master.head(3)

In [ ]:
# Anomaly summary
print('Impossible deliveries:', master['flag_impossible_delivery'].sum())
print('Early reviews:', master['flag_early_review'].sum())
print('Null delivery dates (non-delivered):', master['order_delivered_customer_date'].isnull().sum())
print('Delivery delay range (days):', master['delivery_delay'].min(), 'to', master['delivery_delay'].max())

## 4. Exploratory Data Analysis

In [ ]:
# --- 4.1 Order volume over time ---
monthly = (
    master.set_index('order_purchase_timestamp')
    .resample('ME')['order_id']
    .count()
    .reset_index()
)
monthly.columns = ['month', 'orders']

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(monthly['month'], monthly['orders'], marker='o', lw=2)
ax.set(title='Monthly Order Volume (2016–2018)', xlabel='', ylabel='Orders')
plt.tight_layout()
plt.savefig(OUTPUTS + '01_monthly_orders.png', dpi=150)
plt.show()

In [ ]:
# --- 4.2 Orders by state ---
state_orders = master['customer_state'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 5))
state_orders.plot(kind='bar', ax=ax, color='steelblue')
ax.set(title='Top 15 States by Order Volume', xlabel='State', ylabel='Orders')
plt.tight_layout()
plt.savefig(OUTPUTS + '02_orders_by_state.png', dpi=150)
plt.show()

In [ ]:
# --- 4.3 Top categories by revenue ---
cat_rev = (
    master.groupby('product_category_name_english')['total_price']
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(10, 5))
cat_rev.plot(kind='bar', ax=ax, color='coral')
ax.set(title='Top 15 Categories by Revenue (BRL)', xlabel='', ylabel='Revenue')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1e6:.1f}M'))
plt.tight_layout()
plt.savefig(OUTPUTS + '03_revenue_by_category.png', dpi=150)
plt.show()

In [ ]:
# --- 4.4 Review score distribution ---
fig, ax = plt.subplots(figsize=(7, 4))
master['review_score'].value_counts().sort_index().plot(kind='bar', ax=ax, color='mediumpurple')
ax.set(title='Review Score Distribution', xlabel='Score', ylabel='Count')
plt.tight_layout()
plt.savefig(OUTPUTS + '04_review_scores.png', dpi=150)
plt.show()

In [ ]:
# --- 4.5 Payment installments ---
fig, ax = plt.subplots(figsize=(10, 4))
master['payment_installments'].value_counts().sort_index().head(12).plot(kind='bar', ax=ax, color='teal')
ax.set(title='Payment Installments Distribution', xlabel='Installments', ylabel='Orders')
plt.tight_layout()
plt.savefig(OUTPUTS + '05_installments.png', dpi=150)
plt.show()

## 5. Angle 1 — Delivery Performance

In [ ]:
delivered = filter_delivered(master)

on_time_pct = (delivered['delivery_delay'] <= 0).mean()
print(f'On-time delivery rate: {on_time_pct:.1%}')
print(f'Median delay: {delivered["delivery_delay"].median():.1f} days')
print(f'Mean delay: {delivered["delivery_delay"].mean():.1f} days')

In [ ]:
# Delivery delay distribution
fig, ax = plt.subplots(figsize=(10, 4))
delivered['delivery_delay'].clip(-20, 60).hist(bins=60, ax=ax, color='steelblue', edgecolor='white')
ax.axvline(0, color='red', lw=2, linestyle='--', label='On time')
ax.set(title='Delivery Delay Distribution (days)', xlabel='Days (negative = early)', ylabel='Orders')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUTS + '06_delivery_delay_dist.png', dpi=150)
plt.show()

In [ ]:
# Delay by state
state_delay = (
    delivered.groupby('customer_state')['delivery_delay']
    .agg(['mean', 'median', 'count'])
    .sort_values('mean', ascending=False)
    .reset_index()
)
state_delay.columns = ['state', 'mean_delay', 'median_delay', 'n_orders']

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['tomato' if x > 0 else 'seagreen' for x in state_delay['mean_delay']]
ax.bar(state_delay['state'], state_delay['mean_delay'], color=colors)
ax.axhline(0, color='black', lw=0.8)
ax.set(title='Mean Delivery Delay by State (days)', xlabel='State', ylabel='Days')
plt.tight_layout()
plt.savefig(OUTPUTS + '07_delay_by_state.png', dpi=150)
plt.show()

In [ ]:
# Delay vs review score
fig, ax = plt.subplots(figsize=(8, 5))
delivered.groupby('review_score')['delivery_delay'].mean().plot(kind='bar', ax=ax, color='mediumpurple')
ax.axhline(0, color='red', lw=1, linestyle='--')
ax.set(title='Mean Delivery Delay by Review Score', xlabel='Review Score', ylabel='Mean Delay (days)')
plt.tight_layout()
plt.savefig(OUTPUTS + '08_delay_vs_score.png', dpi=150)
plt.show()

## 6. Angle 2 — Customer Segmentation (RFM)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

snapshot_date = master['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = (
    master[master['order_status'] == 'delivered']
    .groupby('customer_unique_id')
    .agg(
        recency=('order_purchase_timestamp', lambda x: (snapshot_date - x.max()).days),
        frequency=('order_id', 'nunique'),
        monetary=('total_payment', 'sum'),
    )
    .reset_index()
)
print(rfm.describe())

In [ ]:
# Elbow method
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['recency', 'frequency', 'monetary']])

inertias = []
K = range(2, 10)
for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(K, inertias, marker='o')
ax.set(title='Elbow Method — KMeans on RFM', xlabel='k', ylabel='Inertia')
plt.tight_layout()
plt.show()

In [ ]:
# Fit final model (adjust k based on elbow plot)
K_FINAL = 4
km = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
rfm['cluster'] = km.fit_predict(rfm_scaled)

segment_profile = rfm.groupby('cluster')[['recency', 'frequency', 'monetary']].mean().round(1)
segment_profile['n_customers'] = rfm.groupby('cluster').size()
print(segment_profile)

In [ ]:
# Assign human-readable segment names based on profile
# (update mapping after inspecting segment_profile above)
SEGMENT_NAMES = {0: 'Champions', 1: 'Loyal', 2: 'At-Risk', 3: 'Lost'}
rfm['segment'] = rfm['cluster'].map(SEGMENT_NAMES)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, col in zip(axes, ['recency', 'frequency', 'monetary']):
    rfm.groupby('segment')[col].mean().sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Mean {col.capitalize()} by Segment')
plt.tight_layout()
plt.savefig(OUTPUTS + '09_rfm_segments.png', dpi=150)
plt.show()

## 7. Angle 3 — Review Score Prediction

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay
import shap

# Binary target: bad review (1-2) vs good review (4-5); drop neutral (3)
model_df = delivered[delivered['review_score'] != 3].copy()
model_df['bad_review'] = (model_df['review_score'] <= 2).astype(int)
print(f'Class balance — bad: {model_df["bad_review"].mean():.1%}, good: {1-model_df["bad_review"].mean():.1%}')

In [ ]:
NUMERIC_FEATURES = [
    'delivery_delay', 'days_to_deliver', 'days_to_approve',
    'total_price', 'total_freight', 'freight_ratio',
    'payment_installments', 'n_items',
    'product_weight_g', 'product_volume_cm3',
]
CATEGORICAL_FEATURES = [
    'customer_state', 'seller_state',
    'primary_payment_type', 'product_category_name_english',
]

X = model_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = model_df['bad_review']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

In [ ]:
numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])
categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipe, NUMERIC_FEATURES),
    ('cat', categorical_pipe, CATEGORICAL_FEATURES),
])

# Baseline
lr_pipe = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(max_iter=500, random_state=42))])
lr_pipe.fit(X_train, y_train)
print('--- Logistic Regression ---')
print(classification_report(y_test, lr_pipe.predict(X_test)))
print(f'ROC-AUC: {roc_auc_score(y_test, lr_pipe.predict_proba(X_test)[:,1]):.3f}')

In [ ]:
# Random Forest
rf_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)),
])
rf_pipe.fit(X_train, y_train)
print('--- Random Forest ---')
print(classification_report(y_test, rf_pipe.predict(X_test)))
print(f'ROC-AUC: {roc_auc_score(y_test, rf_pipe.predict_proba(X_test)[:,1]):.3f}')

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(rf_pipe, X_test, y_test, ax=ax, display_labels=['Good', 'Bad'])
ax.set_title('Random Forest — Confusion Matrix')
plt.tight_layout()
plt.savefig(OUTPUTS + '10_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# SHAP feature importance
X_test_transformed = rf_pipe['prep'].transform(X_test)
feature_names = (
    NUMERIC_FEATURES
    + rf_pipe['prep'].named_transformers_['cat']['encode'].get_feature_names_out(CATEGORICAL_FEATURES).tolist()
)

explainer = shap.TreeExplainer(rf_pipe['clf'])
shap_values = explainer.shap_values(X_test_transformed[:500])  # sample for speed

shap.summary_plot(shap_values[1], X_test_transformed[:500], feature_names=feature_names,
                  max_display=15, show=False)
plt.tight_layout()
plt.savefig(OUTPUTS + '11_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Findings & Recommendations

### Delivery Performance
- *Fill in after analysis*

### Customer Segments
- *Fill in after analysis*

### Review Prediction
- *Fill in after analysis*

### Actionable Recommendations
1. *Fill in after analysis*
2. *Fill in after analysis*
3. *Fill in after analysis*